# Clustering v2

Version: educloud-clustering-v2.0.0. Implement `ds.py` before running this notebook. The bundled data are fictional: use them for workflow practice, not conclusions about countries or energy policy. An instructor may supply a reviewed OWID CSV with the same feature columns. See README.md and feedback.md for the single 100-point rubric; this notebook is formative and has no extra points.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import ds

raw = ds.load_energy_data("data/energy-synthetic.csv")
features = ["fossil_energy_per_capita", "renewables_energy_per_capita", "low_carbon_share_energy"]
clean = ds.clean_energy_data(raw, subset_cols=features)
print(ds.initial_summary(clean))

## Separate training, calibration and evaluation

Split before fitting either scaling parameters or models. This random split is for the independent synthetic rows. Real repeated country/year records require a grouped or time-based split chosen before analysis. Fit scaling only on training data, then apply those same parameters to both other sets.

In [ ]:
train, remaining = train_test_split(clean, test_size=0.4, random_state=42)
reference, evaluation = train_test_split(remaining, test_size=0.5, random_state=42)
train_features = ds.select_clustering_features(train, features)
X_train, means, raw_stds = ds.standardize_features(train_features)
divisors = np.where(raw_stds == 0, 1.0, raw_stds)
X_reference = (ds.select_clustering_features(reference, features) - means) / divisors
X_evaluation = (ds.select_clustering_features(evaluation, features) - means) / divisors
assert set(train.index).isdisjoint(reference.index)
assert set(train.index).isdisjoint(evaluation.index)
assert set(reference.index).isdisjoint(evaluation.index)

## Compare clustering methods on training data

Choose K using training data and explain your choice. Compare CH and silhouette scores without interpreting either as a universal measure of usefulness.

In [ ]:
ch_scores = ds.compute_calinski_harabasz_scores(X_train, range(2, 5))
k = max(ch_scores, key=ch_scores.get)
labels, centers = ds.run_kmeans(X_train, k)
hierarchical = ds.run_hierarchical_clustering(X_train, k, linkage="average")
print(ch_scores, ds.silhouette_for_labels(X_train, labels))
model = ds.fit_gmm(X_train, k)
labels_evaluation = ds.gmm_cluster_labels(model, X_evaluation)

## Calibrate a cutoff, then evaluate

Freeze a cutoff using a reference set that was not used to fit the GMM. The default tail fraction is 0.01. It is not a promise that exactly 1% of future observations will be flagged; small sets and ties make that particularly misleading. The inequality is strict: equality is not flagged. Do not recompute this cutoff on the evaluation batch.

In [ ]:
log_density_cutoff = ds.calibrate_outlier_cutoff(model, X_reference)
outliers = ds.gmm_outlier_mask(model, X_evaluation, log_density_cutoff=log_density_cutoff)
result = evaluation.copy()
result["cluster"] = labels_evaluation
result["low_density"] = outliers
print(result)
first_alone = ds.gmm_outlier_mask(model, X_evaluation.iloc[:1], log_density_cutoff=log_density_cutoff)
assert first_alone[0] == outliers[0]

Explain why the same observation keeps its label when evaluated alone. Discuss the effect of a small reference set, a shifted evaluation population, and tied log densities. Explain why density is not a probability mass and why exponentiating it does not create a number restricted to [0, 1].

## Optional: membership uncertainty

Implement `membership_uncertainty_mask` if you want this unscored extension. A point may be confidently assigned to one component yet be far from every component. Conversely, a high-density point between components can have uncertain membership. With K components, the maximum normalized responsibility is at least 1/K; a lower confidence threshold can never flag a point.

In [ ]:
# Uncomment after implementing the optional function.
# uncertain = ds.membership_uncertainty_mask(model, X_evaluation, min_confidence=0.9)
# print(pd.crosstab(outliers, uncertain, rownames=["low density"], colnames=["uncertain membership"]))